# **04 - Conformal Prediction Application**

## **Summary**

---
## **Table of Contents**

* [Summary](#summary)

* [Objective](#objective)

* [Dataset Info](#dataset-info)

* [Initial Setup](#initial-setup)

* [Import Model](#import-model)

* [Apply Conformal Prediction](#apply-conformal-prediction)

---
## **Objective**
To produce an efficient conformal prediction module with the goal of producing a narrow interval that covers 95% of the actual electricity consumption. This module is to be integrated with our predictive model in the deployment stage.

## <a id ='data-info'></a> **Dataset Info**
* **Source:** UCI Machine Learning Repository
* **Title:** Appliances Energy Prediction
* **Time Range:** 11th January 2016 - 27th May 2016
* **Primary Target Variable:** `Appliances` (Watt-hour consumed)

## **Initial Setup**

In [ ]:
import pandas as pd
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split
import joblib
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from mapie.regression import SplitConformalRegressor

# Add the root directory (one level up from notebooks/) to Python's path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.preprocess import preprocess_energy, feature_engineering, xgb_preprocess

In [ ]:
PROJECT_ROOT = Path.cwd().parent

# Import csv file
energy = pd.read_csv(f'{PROJECT_ROOT}/data/raw/energydata_complete.csv')

In [ ]:
# Data preprocessing
energy = preprocess_energy(energy)

# Form feature through feature engineering
energy = feature_engineering(energy)

ENCODER_PATH = root_dir / "models" / "month_encoder.pkl"

encoder = joblib.load(ENCODER_PATH)

# Separate feature from target and preprocess predictors
X = xgb_preprocess(energy, encoder)
y = energy['Appliances']


# Train-calibration-test split (70-15-15)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, shuffle= False
)

X_cal, X_test, y_cal, y_test = train_test_split(
    X_test, y_test, shuffle= False
)

## **Import Model**

In [ ]:
MODEL_PATH = PROJECT_ROOT/"models"/"xgb_best.pkl"

# Use joblib.load instead of pickle.load
model = joblib.load(MODEL_PATH)

## **Apply Conformal Prediction**

In [ ]:
mapie_regressor = SplitConformalRegressor(
    estimator= model,
    confidence_level= 0.95,
    prefit = True
)

mapie_regressor.conformalize(X_cal, y_cal)

y_pred, y_intervals = mapie_regressor.predict_interval(X_test)

In [ ]:
# Ensure 2D shape [N, 2] for bounds
y_intervals = np.squeeze(y_intervals)

# Build DataFrame using X_test's DateTime index
results_df= pd.DataFrame({
    'actual': np.squeeze(y_test),
    'prediction': np.squeeze(y_pred),
    'lower_bound': y_intervals[:, 0],
    'upper_bound': y_intervals[:, 1]
}, index=X_test.index)

for t in range(len(y_test)):
    results_df.iloc[t, 2] = max(0, results_df.iloc[t,2])


In [ ]:
missed = results_df[
    (results_df['actual'] < results_df['lower_bound']) | 
    (results_df['actual'] > results_df['upper_bound'])
]

In [ ]:
(results_df['upper_bound'] - results_df['lower_bound']).mean()

In [ ]:
# 2. Plotting
plt.figure(figsize=(14, 6))

# Plot Actuals and Predictions
plt.plot(results_df.index, results_df['actual'], label='Actual', color='gray', linewidth=1.5, linestyle = '--')
plt.plot(results_df.index, results_df['prediction'], label='Prediction', color='black', linewidth=1.5)

# Fill prediction interval
plt.fill_between(
    results_df.index,
    results_df['lower_bound'],
    results_df['upper_bound'],
    color='tab:blue',
    alpha=0.25,
    label='Prediction Interval'
)

# Scatter missed observations
plt.scatter(
    x=missed.index,
    y=missed['actual'],
    color='crimson',
    marker="x",
    s=60,
    linewidths=2,
    label="Missed observations",
    zorder=4
)

plt.title("Conformal Prediction Intervals Over Time")
plt.xlabel("Date")
plt.ylabel("Value")
plt.legend(loc="upper left")
plt.grid(True, linestyle='--', alpha=0.5)

# Automatically format date axis labels
plt.gcf().autofmt_xdate()

plt.tight_layout()
plt.show()

> ### Observations
>
> ```Intervals capture most observations during stable periods.```
>
> ```Extreme spikes is not captured well```
>
> But this is a model weakness (our model underpredicting heavy spikes), not conformal prediction weakness.